# NightClear — Full Training Pipeline (Google Colab, from scratch)

Self-contained: the network, losses, data pipeline and training loop are all defined in this notebook (no repo code needed).

1. Setup (packages, Google Drive, GPU)
2. Configuration
3. Data: unzip, pairing, augmentation, sanity check
4. Model: NightClear (DarkIR architecture) built from scratch
5. Loss: `L = L1 + 0.01·LPIPS + 50·Edge + LowRes`
6. Optimizer, scheduler, resume
7. Training loop (mixed precision, checkpoints every epoch)
8. Training curves
9. Final evaluation (PSNR / SSIM / LPIPS)
10. Visual results
11. Export weights for the web app

### Before you start
1. **Runtime → Change runtime type → GPU.**
2. Zip the dataset on your computer (`cd data/datasets && zip -r LOL-v2.zip LOL-v2`) and upload it to Google Drive as **`MyDrive/NightClear/LOL-v2.zip`**.
   Expected layout inside: `LOL-v2/{Real_captured,Synthetic}/{Train,Test}/{Low,Normal}`.

## 1. Setup

In [ ]:
%pip install -q lpips kornia

In [ ]:
import json, random, shutil, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import lpips
import kornia

from google.colab import drive
drive.mount('/content/drive')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device :', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(no GPU: switch the runtime to GPU)')
print('PyTorch:', torch.__version__)

## 2. Configuration

All knobs in one place. The paper trains with batch 32 on an H100; batch 8 at 384×384 crops is meant for a 16 GB Colab GPU (T4 / L4). If you run out of memory, lower `batch_size` first, then `crop` (keep it a multiple of 8).

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/NightClear'

CFG = dict(
    # data
    data_zip    = f'{DRIVE_DIR}/LOL-v2.zip',
    local_data  = '/content/data',   # unzipped here: local disk is much faster than Drive
    subsets     = ['Real_captured', 'Synthetic'],
    train_per_subset = 100,           # None = all training pairs
    test_per_subset  = 50,            # None = all test pairs
    crop        = 384,
    batch_size  = 8,
    num_workers = 2,                  # Colab has 2 CPU cores
    # optimisation
    epochs       = 20,
    lr           = 5e-4,
    eta_min      = 1e-6,
    weight_decay = 1e-4,
    betas        = (0.9, 0.999),
    amp          = True,              # mixed precision; the FFT blocks run in fp32 internally
    # loss weights: L = L1 + 0.01*LPIPS + 50*Edge + LowRes
    w_l1 = 1.0, w_lpips = 0.01, w_edge = 50.0, w_lowres = 1.0,
    # network (NightClear-m: width 32, ~3.3 M params)
    net = dict(img_channel=3, width=32, middle_blk_num_enc=2, middle_blk_num_dec=2,
               enc_blk_nums=[1, 2, 3], dec_blk_nums=[3, 1, 1],
               dilations=[1, 4, 9], extra_depth_wise=True),
    # bookkeeping
    eval_every  = 5,                  # epochs between test-set evaluations
    seed        = 42,
    run_dir     = '/content/run',
    export_path = '/content/NightClear.pt',
    resume      = False,              # True: continue from run_dir/last.pt if it exists
)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])
torch.backends.cudnn.benchmark = True
RUN_DIR = Path(CFG['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'config.json').write_text(json.dumps(CFG, indent=2))

## 3. Data

The zip is extracted once per Colab session to local disk. Real images are named `lowXXXXX.png` ↔ `normalXXXXX.png`, Synthetic pairs share the same name, so files are paired by name with the `low` / `normal` prefix stripped.

In [ ]:
LOCAL_DATA = Path(CFG['local_data'])
if not list(LOCAL_DATA.glob('**/Real_captured')):
    zip_path = Path(CFG['data_zip'])
    assert zip_path.exists(), f'Upload the dataset zip to {zip_path} first'
    print('Unzipping', zip_path, '->', LOCAL_DATA, '...')
    shutil.unpack_archive(zip_path, LOCAL_DATA)
DATA_ROOT = next(LOCAL_DATA.glob('**/Real_captured')).parent
print('Dataset root:', DATA_ROOT)

def pair_files(subset, split):
    '''Return sorted [(low_path, normal_path), ...] for one subset and split ("Train" / "Test").'''
    base = DATA_ROOT / subset / split
    key = lambda p: p.name.removeprefix('low').removeprefix('normal')
    lows    = {key(p): p for p in (base / 'Low').glob('*.png')}
    normals = {key(p): p for p in (base / 'Normal').glob('*.png')}
    assert lows, f'no images found in {base}/Low'
    assert lows.keys() == normals.keys(), f'unpaired files in {base}'
    return [(lows[k], normals[k]) for k in sorted(lows)]

rng = random.Random(CFG['seed'])
def subsample(pairs, n):
    '''Fixed random subset of n pairs (all pairs if n is None).'''
    return rng.sample(pairs, n) if n and n < len(pairs) else pairs

train_by_subset = {s: subsample(pair_files(s, 'Train'), CFG['train_per_subset']) for s in CFG['subsets']}
test_pairs      = {s: subsample(pair_files(s, 'Test'), CFG['test_per_subset']) for s in CFG['subsets']}
train_pairs     = [p for s in CFG['subsets'] for p in train_by_subset[s]]

pd.DataFrame({s: {'train': len(train_by_subset[s]), 'test': len(test_pairs[s])} for s in CFG['subsets']}).T

In [ ]:
def load_rgb(path):
    '''PNG -> float tensor (3, H, W) in [0, 1].'''
    return torch.from_numpy(np.asarray(Image.open(path).convert('RGB'), dtype=np.float32) / 255.).permute(2, 0, 1)

class PairedDataset(Dataset):
    '''Low/normal pairs. With crop: random crop + flips + 90° rotations (same op on both). Without: full image.'''
    def __init__(self, pairs, crop=None):
        self.pairs, self.crop = pairs, crop

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        low, high = (load_rgb(p) for p in self.pairs[i])
        if self.crop:
            c = self.crop
            _, h, w = low.shape
            if h < c or w < c:  # pad small images so the crop always fits
                pad = (0, max(0, c - w), 0, max(0, c - h))
                low, high = F.pad(low, pad, mode='reflect'), F.pad(high, pad, mode='reflect')
                _, h, w = low.shape
            y, x = random.randint(0, h - c), random.randint(0, w - c)
            low, high = low[:, y:y + c, x:x + c], high[:, y:y + c, x:x + c]
            if random.random() < 0.5:
                low, high = low.flip(-1), high.flip(-1)
            if random.random() < 0.5:
                low, high = low.flip(-2), high.flip(-2)
            k = random.randint(0, 3)
            low, high = torch.rot90(low, k, (1, 2)), torch.rot90(high, k, (1, 2))
        return low, high

train_loader = DataLoader(PairedDataset(train_pairs, CFG['crop']), batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True,
                          persistent_workers=CFG['num_workers'] > 0)
test_loaders = {s: DataLoader(PairedDataset(p), batch_size=1, num_workers=CFG['num_workers'], pin_memory=True)
                for s, p in test_pairs.items()}
print(f'{len(train_pairs)} training pairs -> {len(train_loader)} iterations / epoch')

Sanity check: one augmented batch. Low-light crops should look almost black, ground truth normally exposed.

In [ ]:
low, high = next(iter(train_loader))
print('batch:', tuple(low.shape), f'| mean brightness low {low.mean():.3f} vs normal {high.mean():.3f}')

n = min(4, len(low))
fig, axes = plt.subplots(2, n, figsize=(3.5 * n, 7), squeeze=False)
for j in range(n):
    axes[0, j].imshow(low[j].permute(1, 2, 0));  axes[0, j].set_title('Low-light')
    axes[1, j].imshow(high[j].permute(1, 2, 0)); axes[1, j].set_title('Ground truth')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Model — NightClear (DarkIR architecture)

* **EBlock (encoder):** depth-wise conv + channel attention, then **FreMLP**: an MLP on the Fourier *amplitude* (≈ illumination) while the phase (≈ structure) is kept → brightens the image.
* **DBlock (decoder):** parallel **dilated** depth-wise convs (dilation 1, 4, 9) → large receptive field → removes blur.
* **Side output:** a 1/8-resolution lit image predicted at the bottleneck, supervised by the low-res loss.

Layer names match the original DarkIR code, so the exported weights load directly into the web app.

In [ ]:
class LayerNorm2d(nn.Module):
    '''LayerNorm over the channel dimension of an NCHW tensor.'''
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(c))
        self.bias = nn.Parameter(torch.zeros(c))
        self.eps = eps

    def forward(self, x):
        mu = x.mean(1, keepdim=True)
        var = (x - mu).pow(2).mean(1, keepdim=True)
        return self.weight.view(1, -1, 1, 1) * (x - mu) / (var + self.eps).sqrt() + self.bias.view(1, -1, 1, 1)

class Seq(nn.Module):
    '''Sequential container stored as `modules_list` (keeps weight names identical to DarkIR checkpoints).'''
    def __init__(self, *mods):
        super().__init__()
        self.modules_list = nn.ModuleList(mods)

    def forward(self, x):
        for m in self.modules_list:
            x = m(x)
        return x

class SimpleGate(nn.Module):
    def forward(self, x):
        a, b = x.chunk(2, dim=1)
        return a * b

class Branch(nn.Module):
    '''One depth-wise 3x3 convolution with a given dilation.'''
    def __init__(self, c, dilation=1):
        super().__init__()
        self.branch = nn.Sequential(nn.Conv2d(c, c, 3, padding=dilation, groups=c, dilation=dilation))

    def forward(self, x):
        return self.branch(x)

class FreMLP(nn.Module):
    '''MLP on the Fourier amplitude; the phase is kept.'''
    def __init__(self, nc, expand=2):
        super().__init__()
        self.process1 = nn.Sequential(nn.Conv2d(nc, expand * nc, 1), nn.LeakyReLU(0.1, inplace=True),
                                      nn.Conv2d(expand * nc, nc, 1))

    @torch.autocast('cuda', enabled=False)  # cuFFT in fp16 only supports power-of-2 sizes
    def forward(self, x):
        x = x.float()
        _, _, H, W = x.shape
        freq = torch.fft.rfft2(x, norm='backward')
        mag, pha = self.process1(torch.abs(freq)), torch.angle(freq)
        freq = torch.complex(mag * torch.cos(pha), mag * torch.sin(pha))
        return torch.fft.irfft2(freq, s=(H, W), norm='backward')

class EBlock(nn.Module):
    '''Encoder block: spatial mixing + FreMLP (illumination in the frequency domain).'''
    def __init__(self, c, extra_depth_wise=True):
        super().__init__()
        dw = 2 * c
        self.extra_conv = nn.Conv2d(c, c, 3, padding=1, groups=c) if extra_depth_wise else nn.Identity()
        self.conv1 = nn.Conv2d(c, dw, 1)
        self.branches = nn.ModuleList([Branch(dw, dilation=1)])
        self.sca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(dw // 2, dw // 2, 1))
        self.sg1 = SimpleGate()
        self.conv3 = nn.Conv2d(dw // 2, c, 1)
        self.norm1, self.norm2 = LayerNorm2d(c), LayerNorm2d(c)
        self.freq = FreMLP(c, expand=2)
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, inp):
        x = self.conv1(self.extra_conv(self.norm1(inp)))
        z = self.sg1(sum(b(x) for b in self.branches))
        y = inp + self.beta * self.conv3(self.sca(z) * z)
        return y + y * self.freq(self.norm2(y)) * self.gamma

class DBlock(nn.Module):
    '''Decoder block: parallel dilated depth-wise convs (large receptive field -> deblurring) + gated FFN.'''
    def __init__(self, c, dilations=(1, 4, 9), extra_depth_wise=True):
        super().__init__()
        dw = 2 * c
        self.conv1 = nn.Conv2d(c, dw, 1)
        self.extra_conv = nn.Conv2d(dw, dw, 3, padding=1, groups=c) if extra_depth_wise else nn.Identity()
        self.branches = nn.ModuleList([Branch(dw, dilation=d) for d in dilations])
        self.sca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(dw // 2, dw // 2, 1))
        self.sg1, self.sg2 = SimpleGate(), SimpleGate()
        self.conv3 = nn.Conv2d(dw // 2, c, 1)
        self.conv4 = nn.Conv2d(c, 2 * c, 1)
        self.conv5 = nn.Conv2d(c, c, 1)
        self.norm1, self.norm2 = LayerNorm2d(c), LayerNorm2d(c)
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, inp):
        x = self.extra_conv(self.conv1(self.norm1(inp)))
        z = self.sg1(sum(b(x) for b in self.branches))
        y = inp + self.beta * self.conv3(self.sca(z) * z)
        x = self.conv5(self.sg2(self.conv4(self.norm2(y))))
        return y + x * self.gamma

class NightClear(nn.Module):
    '''U-shaped encoder-decoder: the encoder fixes illumination, the decoder removes blur.'''
    def __init__(self, img_channel=3, width=32, middle_blk_num_enc=2, middle_blk_num_dec=2,
                 enc_blk_nums=(1, 2, 3), dec_blk_nums=(3, 1, 1), dilations=(1, 4, 9), extra_depth_wise=True):
        super().__init__()
        self.intro = nn.Conv2d(img_channel, width, 3, padding=1)
        self.ending = nn.Conv2d(width, img_channel, 3, padding=1)
        self.encoders, self.downs = nn.ModuleList(), nn.ModuleList()
        self.decoders, self.ups = nn.ModuleList(), nn.ModuleList()

        chan = width
        for n in enc_blk_nums:
            self.encoders.append(Seq(*[EBlock(chan, extra_depth_wise) for _ in range(n)]))
            self.downs.append(nn.Conv2d(chan, 2 * chan, 2, 2))
            chan *= 2
        self.middle_blks_enc = Seq(*[EBlock(chan, extra_depth_wise) for _ in range(middle_blk_num_enc)])
        self.middle_blks_dec = Seq(*[DBlock(chan, dilations, extra_depth_wise) for _ in range(middle_blk_num_dec)])
        self.side_out = nn.Conv2d(chan, img_channel, 3, padding=1)  # 1/8-res lit image for the low-res loss
        for n in dec_blk_nums:
            self.ups.append(nn.Sequential(nn.Conv2d(chan, chan * 2, 1, bias=False), nn.PixelShuffle(2)))
            chan //= 2
            self.decoders.append(Seq(*[DBlock(chan, dilations, extra_depth_wise) for _ in range(n)]))
        self.padder_size = 2 ** len(self.encoders)

    def forward(self, inp, side_loss=False):
        _, _, H, W = inp.shape
        inp = F.pad(inp, (0, (-W) % self.padder_size, 0, (-H) % self.padder_size))  # pad to a multiple of 8
        x = self.intro(inp)
        skips = []
        for enc, down in zip(self.encoders, self.downs):
            x = enc(x)
            skips.append(x)
            x = down(x)
        x_light = self.middle_blks_enc(x)               # end of the encoder: illumination fixed
        x = self.middle_blks_dec(x_light) + x_light
        for dec, up, skip in zip(self.decoders, self.ups, skips[::-1]):
            x = dec(up(x) + skip)
        out = (self.ending(x) + inp)[:, :, :H, :W]      # global residual, crop the padding
        return (self.side_out(x_light), out) if side_loss else out

model = NightClear(**CFG['net']).to(DEVICE)
print(f'NightClear parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M')
with torch.no_grad():
    side, out = model(torch.rand(1, 3, 256, 256, device=DEVICE), side_loss=True)
print('full-res output:', tuple(out.shape), '| low-res side output:', tuple(side.shape))

## 5. Loss

| Term | Weight | Purpose |
|---|---|---|
| L1(output, GT) | 1 | pixel fidelity |
| LPIPS-VGG(output, GT) | 0.01 | perceptual quality |
| Edge (Laplacian pyramid, MSE) | 50 | sharp edges → deblurring |
| L1(side output, GT ↓8) | 1 | forces the encoder to fix illumination |

The loss is computed in fp32 (outside autocast) for stability.

In [ ]:
class EdgeLoss(nn.Module):
    '''MSE between the Laplacian (high-frequency) maps of output and target.'''
    def __init__(self):
        super().__init__()
        k = torch.tensor([[.05, .25, .4, .25, .05]])
        self.register_buffer('kernel', (k.t() @ k).unsqueeze(0).repeat(3, 1, 1, 1))

    def gauss(self, x):
        return F.conv2d(F.pad(x, (2, 2, 2, 2), mode='replicate'), self.kernel, groups=3)

    def laplacian(self, x):
        f = self.gauss(x)
        up = torch.zeros_like(f)
        up[:, :, ::2, ::2] = f[:, :, ::2, ::2] * 4
        return x - self.gauss(up)

    def forward(self, x, y):
        return F.mse_loss(self.laplacian(x), self.laplacian(y))

class NightClearLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.lpips = lpips.LPIPS(net='vgg', verbose=False).eval()
        for p in self.lpips.parameters():
            p.requires_grad_(False)
        self.edge = EdgeLoss()

    def forward(self, out, side, gt):
        gt_low = F.interpolate(gt, size=side.shape[-2:], mode='area')
        parts = {
            'l1':     F.l1_loss(out, gt),
            'lpips':  self.lpips(out, gt, normalize=True).mean(),  # normalize: [0,1] -> [-1,1]
            'edge':   self.edge(out, gt),
            'lowres': F.l1_loss(side, gt_low),
        }
        total = (CFG['w_l1'] * parts['l1'] + CFG['w_lpips'] * parts['lpips']
                 + CFG['w_edge'] * parts['edge'] + CFG['w_lowres'] * parts['lowres'])
        return total, {k: v.item() for k, v in parts.items()}

criterion = NightClearLoss().to(DEVICE)

## 6. Optimizer, scheduler, resume

AdamW with cosine annealing 5e-4 → 1e-6 over all epochs. With `resume=True` and an existing `run_dir/last.pt`, training continues where it stopped (model, optimizer, scheduler, AMP scaler and history are restored).

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'], betas=CFG['betas'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=CFG['eta_min'])
scaler = torch.amp.GradScaler('cuda', enabled=CFG['amp'] and DEVICE.type == 'cuda')

start_epoch, best_psnr, history = 0, 0.0, []
last_ckpt = RUN_DIR / 'last.pt'
if CFG['resume'] and last_ckpt.exists():
    ckpt = torch.load(last_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch, best_psnr, history = ckpt['epoch'], ckpt['best_psnr'], ckpt['history']
    print(f'Resumed from epoch {start_epoch} (best mean PSNR {best_psnr:.2f} dB)')
else:
    print('Training from scratch')

## 7. Training

Every epoch saves `last.pt`; every `eval_every` epochs the model is scored on both test sets and `best.pt` keeps the highest mean PSNR.

In [ ]:
@torch.no_grad()
def evaluate(loader, lpips_fn=None):
    '''Mean PSNR / SSIM (and LPIPS if lpips_fn is given) over full-resolution test images.'''
    model.eval()
    scores = {'psnr': [], 'ssim': []} | ({'lpips': []} if lpips_fn else {})
    for low, high in loader:
        low, high = low.to(DEVICE), high.to(DEVICE)
        with torch.autocast(DEVICE.type, enabled=scaler.is_enabled()):
            out = model(low)
        out = out.float().clamp(0, 1)
        scores['psnr'].append((10 * torch.log10(1 / F.mse_loss(out, high))).item())
        scores['ssim'].append(kornia.metrics.ssim(out, high, window_size=11).mean().item())
        if lpips_fn:
            scores['lpips'].append(lpips_fn(out, high, normalize=True).item())
    model.train()
    return {k: float(np.mean(v)) for k, v in scores.items()}

def save_checkpoint(path, epoch):
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict(), 'best_psnr': best_psnr, 'history': history,
                'config': CFG}, path)

In [ ]:
model.train()
for epoch in range(start_epoch, CFG['epochs']):
    t0, running = time.time(), {}
    for low, high in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{CFG["epochs"]}', leave=False):
        low, high = low.to(DEVICE, non_blocking=True), high.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, enabled=scaler.is_enabled()):
            side, out = model(low, side_loss=True)
        loss, parts = criterion(out.float(), side.float(), high)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        for k, v in {'loss': loss.item(), **parts}.items():
            running[k] = running.get(k, 0.0) + v

    lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    record = {'epoch': epoch + 1, 'lr': lr, 'time_s': time.time() - t0,
              **{k: v / len(train_loader) for k, v in running.items()}}

    if (epoch + 1) % CFG['eval_every'] == 0 or epoch + 1 == CFG['epochs']:
        for s, loader in test_loaders.items():
            record.update({f'{s}/{k}': v for k, v in evaluate(loader).items()})
        mean_psnr = np.mean([record[f'{s}/psnr'] for s in test_loaders])
        if mean_psnr > best_psnr:
            best_psnr = float(mean_psnr)
            save_checkpoint(RUN_DIR / 'best.pt', epoch + 1)

    history.append(record)
    save_checkpoint(last_ckpt, epoch + 1)
    print(' | '.join(f'{k} {v:.4g}' if isinstance(v, float) else f'{k} {v}' for k, v in record.items()))

print(f'Done. Best mean test PSNR: {best_psnr:.2f} dB')

## 8. Training curves

In [ ]:
ep = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for k in ['loss', 'l1', 'lpips', 'edge', 'lowres']:
    axes[0].plot(ep, [h[k] for h in history], label=k)
axes[0].set_yscale('log'); axes[0].set_title('Training loss (total + unweighted terms)')
axes[0].set_xlabel('epoch'); axes[0].legend()

for ax, m in zip(axes[1:], ['psnr', 'ssim']):
    for s in CFG['subsets']:
        pts = [(h['epoch'], h[f'{s}/{m}']) for h in history if f'{s}/{m}' in h]
        if pts:
            ax.plot(*zip(*pts), marker='o', label=s)
    ax.set_title(f'Test {m.upper()}'); ax.set_xlabel('epoch'); ax.legend()

plt.tight_layout(); plt.savefig(RUN_DIR / 'curves.png', dpi=150); plt.show()

## 9. Final evaluation

Reload the best checkpoint and score it on both test sets. LPIPS here uses the standard AlexNet backbone (lower is better).

In [ ]:
best = torch.load(RUN_DIR / 'best.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(best['model_state_dict'])
print(f"Best checkpoint: epoch {best['epoch']}")

lpips_alex = lpips.LPIPS(net='alex', verbose=False).to(DEVICE).eval()
results = {s: evaluate(loader, lpips_alex) for s, loader in test_loaders.items()}
table = pd.DataFrame(results).T.rename(columns=str.upper).round(4)
table.to_csv(RUN_DIR / 'test_metrics.csv')
table

## 10. Visual results

In [ ]:
@torch.no_grad()
def restore(low):
    model.eval()
    return model(low.unsqueeze(0).to(DEVICE)).clamp(0, 1)[0].cpu()

samples = [p for s in CFG['subsets'] for p in random.sample(test_pairs[s], 2)]
fig, axes = plt.subplots(len(samples), 3, figsize=(13, 3.6 * len(samples)), squeeze=False)
for row, (lp, hp) in zip(axes, samples):
    low, high = load_rgb(lp), load_rgb(hp)
    for ax, img, title in zip(row, [low, restore(low), high], ['Low-light input', 'NightClear', 'Ground truth']):
        ax.imshow(img.permute(1, 2, 0)); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.savefig(RUN_DIR / 'samples.png', dpi=120); plt.show()

## 11. Export for the web app

Saves a plain `state_dict` to `export_path` and checks it reloads. Download it from the Colab file panel, copy it into the repo as `models/NightClear.pt` and point `DarkIRBackend.checkpoint` in `model_zoo/__init__.py` to it.

In [ ]:
torch.save(model.state_dict(), CFG['export_path'])
print('Exported weights ->', CFG['export_path'])

check = NightClear(**CFG['net'])
check.load_state_dict(torch.load(CFG['export_path'], map_location='cpu'), strict=True)
print('Reload check OK')